# Training a UNET with CoredexMLbench data

In [1]:
import datetime
import pathlib
import sys
import os
from copy import deepcopy

In [2]:
import xarray
import numpy

In [3]:
import matplotlib

In [4]:
import torch.utils.data
from torch.optim.lr_scheduler import OneCycleLR

In [5]:
# add cnrm code to python path
sys.path.append('/home/users/shaddad/prog/aids_fork/src/cnrm-unet/src')

In [6]:
import data as cnrm_data
import model as cnrm_model
import train as cnrm_train
import visualisation as cnrm_vis

In [7]:
#todo set up cordexbench training venv for notebook, add to repo

In [8]:
cordexbench_root = pathlib.Path('/gws/nopw/j04/mohc_shared/cordexbench')
print(cordexbench_root.is_dir())
cordexbench_root

True


PosixPath('/gws/nopw/j04/mohc_shared/cordexbench')

In [9]:
[d1 for d1 in cordexbench_root.iterdir() if d1.is_dir()]


[PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/ALPS_domain'),
 PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/NZ_domain'),
 PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/SA_domain')]

In [10]:
domain_names = {
    'alps': 'ALPS_domain',
    'nz': 'NZ_domain',
    'sa': 'SA_domain',
}

In [11]:
domain_dirs = {domain_key: (cordexbench_root / domain_name) for domain_key,domain_name in domain_names.items()}
domain_dirs

{'alps': PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/ALPS_domain'),
 'nz': PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/NZ_domain'),
 'sa': PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/SA_domain')}

Notes for hackathon
* local context is import
* e.g. durban floods - what can climate models show us
* new platforms and new techniques
* can AI match statistical techniques?
* early or late onset of rain? Summer rainfall (agricultural impacts)
* heatwaves,impact on food security
* confident in training their own models so they have their own "lego blocks" for future ML/AI work - model transparency


In [12]:
SA_predictors = ['t_850','z_500','q_850']
SA_target = ['pr']

reference_period = {
    'start': datetime.datetime(1961,1,1,0,0),
    'end': datetime.datetime(2099,1,1,0,0),
}

In [13]:
# Loss function. Must be one of 'mse', 'mae', 'emulasym'.
loss = 'mse'
model_channels = 64
channel_mult = [1, 2, 4, 8, 8]

# Resolution to use for the input data. Currently, only 16 and 64 are supported.
input_resolution = 16
output_resolution = 128
output_channels = 1

epochs = 10
batch_size = 32

# Optimiser type: 'adam' or 'sgd' (default: adam)
optimiser_type = 'adam'

learning_rate = 5e-4
scheduler_type = 'onecycle'
scheduler_step_size = 10
scheduler_gamma = 0.1,
scheduler_max_lr = 5e-4

In [14]:
user_dir = pathlib.Path('/gws/nopw/j04/mohc_shared/users/shaddad/')

In [15]:
!ls /gws/nopw/j04/mohc_shared/users/shaddad/coredex_out


In [16]:
out_path = user_dir / 'cordexbench'
if not out_path.is_dir():
    out_path.mkdir()
    print(f'created dir {out_path}')

In [17]:
domain_dirs = {domain_key: (cordexbench_root / domain_name) for domain_key,domain_name in domain_names.items()}
domain_dirs

{'alps': PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/ALPS_domain'),
 'nz': PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/NZ_domain'),
 'sa': PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/SA_domain')}

In [18]:
list(domain_dirs['sa'].iterdir())

[PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/SA_domain/image_example_SA.png'),
 PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/SA_domain/test'),
 PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/SA_domain/train')]

In [19]:
fname_template_dict = {
    'access_predictors': 'ACCESS-CM2_{start}-{end}.nc',
    'access_target': 'pr_tasmax_ACCESS-CM2_{start}-{end}.nc',

}

In [20]:
! ls /gws/nopw/j04/mohc_shared/cordexbench/SA_domain/test/mid_century/predictors

imperfect  perfect


In [21]:
test_period_dict = {
    'train': (1961,1980),
    'historical': (1981,2000),
    'mid_century': (2041,2060),
    'end_century': (2081,2099)
}

In [22]:
ESD_sa_train_dir = domain_dirs['sa']/'train'/ 'ESD_pseudo_reality'
ESD_sa_static_path = ESD_sa_train_dir / 'predictors' / 'Static_fields.nc'
ESD_sa_train_predictor_path = ESD_sa_train_dir / 'predictors' / fname_template_dict['access_predictors'].format(start=test_period_dict['train'][0],
                                                                                                     end=test_period_dict['train'][1])
ESD_sa_train_target_path = ESD_sa_train_dir / 'target' / fname_template_dict['access_target'].format(start=test_period_dict['train'][0],
                                                                                                     end=test_period_dict['train'][1])


In [23]:
ESD_sa_static_path.is_file(), ESD_sa_train_predictor_path.is_file(), ESD_sa_train_target_path.is_file()

(True, True, True)

In [24]:
test_period = 'mid_century'
test_case = 'perfect'

In [25]:
ESD_sa_test_dir = domain_dirs['sa']/'test'
ESD_sa_test_predictor_path = ESD_sa_test_dir / test_period / 'predictors' / test_case / fname_template_dict['access_predictors'].format(start=test_period_dict[test_period][0],
                                                                                                                                        end=test_period_dict[test_period][1],
                                                                                                                                        )
print(ESD_sa_test_predictor_path.is_file())
ESD_sa_test_predictor_path

True


PosixPath('/gws/nopw/j04/mohc_shared/cordexbench/SA_domain/test/mid_century/predictors/perfect/ACCESS-CM2_2041-2060.nc')

In [26]:
stats_path = pathlib.Path('norm_stats.json')

In [27]:
sa_cordex_train = cnrm_data.CORDEXDataset(
    predictors_data_path=ESD_sa_train_predictor_path,
    predictors=SA_predictors,
    target_data_path=ESD_sa_train_target_path,
    targets=SA_target,
    normalisation_stats_path=stats_path,
    is_train=True,
    reference_period_start=reference_period['start'].year,
    reference_period_end=reference_period['end'].year,
)

In [28]:

sa_cordex_train.normalised_two_d_predictors.shape, sa_cordex_train.target_field.shape
# dir(sa_cordex_train)

((7300, 3, 16, 16), (7300, 128, 128, 1))

In [29]:
#todo: need to split the train data into train and validate as done on azureml, so that we have 

In [30]:
sa_cordex_val = cnrm_data.CORDEXDataset(
    predictors_data_path=ESD_sa_train_predictor_path,
    predictors=SA_predictors,
    target_data_path=ESD_sa_train_target_path,
    targets=SA_target,
    normalisation_stats_path=stats_path,
    is_train=True,
    reference_period_start=reference_period['start'].year,
    reference_period_end=reference_period['end'].year,
)

In [31]:
sa_cordex_train_loader = torch.utils.data.DataLoader(
        sa_cordex_train, batch_size=32, shuffle=True
)
    

In [32]:
sa_cordex_val_loader = torch.utils.data.DataLoader(sa_cordex_val, batch_size=batch_size)


### specifiy training parameters

### training the model


In [33]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [34]:
model = cnrm_model.UNet(
    num_2d_predictors=sa_cordex_train.num_2d_predictors,
    num_1d_predictors=sa_cordex_train.num_1d_predictors,
    model_channels=model_channels,
    channel_mult=channel_mult,
    input_resolution=sa_cordex_train.normalised_two_d_predictors.shape[-1],
    output_resolution= sa_cordex_train.target_field.shape[-2],
    output_channels=output_channels,
).to(DEVICE)


In [35]:

if loss == "mse":
    loss_fn = torch.nn.MSELoss()
elif loss == "mae":
    loss_fn = torch.nn.L1Loss()
elif loss == "emulasym":
    if "pr" not in targets or len(targets) > 1:
        raise ValueError(
            "EmulASYM loss function should only be used for predicting precipitation (pr)."
        )

    # Fit gamma distributions to the training data
    logger.info("Fitting gamma distributions to the training data.")
    alphas, betas = fit_gamma_distributions(
        training_dataset.target_field,
        training_dataset.time,
        reference_period_start,
        reference_period_end,
    )

    np.save(f"{output_dir_path}/alphas.npy", alphas)
    np.save(f"{output_dir_path}/betas.npy", betas)

    alphas = torch.tensor(alphas, dtype=torch.float32).to(DEVICE)
    betas = torch.tensor(betas, dtype=torch.float32).to(DEVICE)

    loss_fn = EmulASYMLoss(alphas, betas)
else:
    raise ValueError(
        f"Invalid loss function '{loss}'. Must be one of 'mse', 'mae', 'emulasym'."
    )


In [36]:
# Optimiser selection
if optimiser_type.lower() == "adam":
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
elif optimiser_type.lower() == "sgd":
    optimiser = torch.optim.SGD(model.parameters(), lr=learning_rate)
else:
    raise ValueError(f"Unsupported optimiser: {optimiser_type}")


In [37]:
# Scheduler selection
if scheduler_type.lower() == "onecycle":
    scheduler = OneCycleLR(
        optimiser,
        max_lr=scheduler_max_lr,
        steps_per_epoch=len(sa_cordex_train_loader),
        epochs=epochs,
    )
elif scheduler_type.lower() == "steplr":
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimiser,
        step_size=scheduler_step_size,
        gamma=scheduler_gamma,
    )
else:
    raise ValueError(f"Unsupported scheduler: {scheduler_type}")

In [39]:
best_val_loss = numpy.inf
for epoch in range(epochs):
    print(f"Epoch {epoch+1}\n-------------------------------")
    cnrm_train.train_loop(
        sa_cordex_train_loader, 
        model, 
        loss_fn, 
        optimiser, 
        scheduler, 
        DEVICE, 
        epoch, 
        scheduler_type
    )

    val_loss = cnrm_train.test_loop(
        sa_cordex_val_loader, 
        model, 
        loss_fn, 
        DEVICE,
        epoch)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = deepcopy(model.state_dict())

    if scheduler is not None:
        print(f"Current learning rate: {scheduler.get_last_lr()[0]}")

    cnrm_vis.log_prediction_visualisation(model, sa_cordex_val, DEVICE, epoch, SA_predictors, SA_target)


Epoch 1
-------------------------------
Current learning rate: 0.0003806340051215103
Epoch 2
-------------------------------
Current learning rate: 0.0004999995198899052
Epoch 3
-------------------------------


KeyboardInterrupt: 

In [ ]:
model_path = f"{output_dir_path}/model.pth"
logger.info(f"Training complete. Saving model to {model_path}")
torch.save(best_model_state, model_path)